# CSC1185 Neural Language Modelling: Week 5 Lab -- Long Short-Term Memory Neural Networks

## 1 Introduction

In the previous lab (Week 4) we implemented a simple Recurrent Neural Network (also known as an Elman RNN), as shown in the figure below:

![](introduction_image1.png)

One challenge with simple RNNs is the **vanishing gradient problem** (see the lecture slides from Weeks 4 and 5). As the network processes longer sequences with many time steps, the gradients during backpropagation can become very small, making it difficult for the network to learn long-term dependencies. This limitation can be addressed by using more advanced architectures like LSTMs (Long Short-Term Memory).


### 1.1 LSTM (Long Short-Term Memory) Networks

LSTMs are a variant of recurrent neural network (RNN) architecture designed to address the vanishing gradient problem, which can occur in traditional RNNs and make it difficult for the network to learn and retain information over long sequences.

LSTMs take as an additional input a memory cell $c_{t-1}$ from the previous time step *t-1*, and produce as an additional output an updated memory cell $c_{t}$ for the current time step *t*. This is illustrated in the following diagram, RNN on the left, and LSTM (leaving aside the internal functionality for the moment) on the right:

![](intro_image2.png)


##### **LSTM List of Inputs:**
1. **Hidden state ($\textbf{h}_{t-1}$):** Captures short-term dependencies within the sequence.
2. **Cell state ($\textbf{c}_{t-1}$):** Captures long-term dependencies within the sequence.
3. **input ($\textbf{x}_t$) :** Individual element of the input sequence (e.g., word, etc.).

Over and above the additional input/output of the memory cell above, LSTMs have a more complex internal structure than the neural networks we have seen so far. In addition to the normal operations multiplying the previous hidden state and the current input with their respective weight matrices to compute (a first version of) the current hidden state g, they perform additional operations on the three inputs above to compute three gates which filter information to be used in the current forward pass, and passed on for use in subsequent forward passes:

##### **LSTM Gates:**
1. **Forget gate f:** To decide which information to delete from the context that is no longer needed.
2. **Input gate (Add gate) i:** To pass on important information we need to extract from the previous hidden state and current inputs to update the cell state c$_{t-1}$.
3. **Output gate o:** To decide what information is required for (the final version of) the current hidden state h$_t$.

The inputs and gates combine as shown in the diagram below (see also lecture slides) to achieve the above filtering effects:

![](intro_image3.png)


## 2 Background and Lab Overview

In this lab, we will focus on implementing the forward pass of an LSTM (Long Short-Term Memory) network from scratch. While we'll go into the details of how the LSTM processes input data and updates its hidden and cell states, we'll rely on PyTorch (Week 3) to handle the rest of the training process. This includes automatically computing the gradients during the backward pass (backpropagation), optimising the model parameters using an optimisation algorithm (e.g., SGD, Adam), and updating the parameters accordingly to minimise the loss function. By implementing the forward pass ourselves, we gain a deeper understanding of how LSTMs work internally, while PyTorch seamlessly manages the training process.
We will use our LSTM implementation in one of the common NLP architectures, a sequence-to-sequence (Seq2Seq) encoder-decoder model, to solve a translation task. For details, refer to the lecture slides and [Speech and Language Processing (3rd ed. draft) Dan Jurafsky and James H. Martin, Chapter 13](https://web.stanford.edu/~jurafsky/slp3/13.pdf)


### 2.1  Sequence-to-Sequence Encoder-Decoder Networks

#### `sequence-to-sequence for intuitive tasks`

Sequence-to-sequence (Seq2Seq) Encoder-Decoder networks ([Sutskever et al.](https://arxiv.org/abs/1409.3215)_) transform a variable-length sequence input to a variable-length sequence output using a fixed-sized model. Next-token/masked-token self-supervised prediction can be viewed as a specialised version of this framework, where the output corresponds to a shifted version of the input or is masked at specific locations.

The vanilla form of a Seq2Seq Encoder-Decoder uses two separate recurrent neural nets together. One RNN (using the term generically, i.e. an RNN could be an Elman RNN, LSTM, etc.) acts as an **encoder**, which encodes a variable-length input sequence ijto a fixed-length context vector. The ideas is that this context vector (the final hidden layer of the first RNN) will contain semantic information about the entire input sequence. The second RNN is a **decoder**, which takes an input word and the context vector z, and returns an output and hidden state to pass on for use in the next time step.

The image below shows an example application to translation:

![](background_1.png)

The input/source sentence, "guten morgen", is passed through the embedding layer (yellow) and then input into the encoder (green). We also append a *start of sequence* (`<sos>`) and *end of sequence* (`<eos>`) token to the start and end of sentence, respectively. At each time step, the inputs to the encoder RNN are the embedding, $e$, of the current word, $e(\textbf{x}_t)$, as well as the hidden state from the previous time-step, $\textbf{h}_{t-1}$. The encoder RNN outputs a new hidden state $h_t$. We can think of each hidden state as a vector representation of the input sentence so far. The RNN can be represented as a function of $e(\textbf{x}_t)$ and $\textbf{h}_{t-1}$:

$$\textbf{h}_t = \text{EncoderRNN}(e(\textbf{x}_t), \textbf{h}_{t-1})$$

Recall that we're using the term RNN generally here, it could be any recurrent architecture, such as an *LSTM* (Long Short-Term Memory) or a *GRU* (Gated Recurrent Unit).

In our current example, we have an input sequence $\textbf{X} = \{\textbf{x}_1, \textbf{x}_2, ..., \textbf{x}_t\}$, where $\textbf{x}_1 = \text{<sos>}, \textbf{x}_2 = \text{guten}$, etc. The initial hidden state, $\textbf{h}_0$, is usually either initialised to zeros or a learned parameter.

Once the final word $\textbf{x}_t$ has been passed into the RNN, we use the final hidden state, $\textbf{h}_t$, as the context vector $\textbf{z}$, i.e. we set $\textbf{z} = \textbf{h}_t$, as a representation of the entire input sentence.

The next step is to pass $\textbf{z}$ to the decoder to generate the translated output sentence, "good morning". We append the start token `<sos>`, and in training, also the end token `<eos>`. At each time step, the inputs to the decoder RNN (blue) are the embedding of the current word $d(\textbf{y}_t)$ (in inference, generated at the previous time step), as well as the hidden state from the previous time-step, $\textbf{s}_{t-1}$, where the initial decoder hidden state, $\textbf{s}_0$, is the context vector, $\textbf{s}_0 = z = \textbf{h}_t$. Thus, similar to the encoder, we can represent the decoder as:

$$\textbf{s}_t = \text{DecoderRNN}(d(\textbf{y}_t), \textbf{s}_{t-1})$$

Although the input embedding layer and the output embedding layer are both shown in yellow in the diagram they are two different embedding layers with their own parameters.

In the decoder, we need to go from the hidden state to an actual word, therefore at each time step we use $\textbf{s}_t$ to predict (by passing it through a linear layer, shown in purple) the next word in the sequence $\hat{\textbf{y}}_t$.

$$\hat{\textbf{y}}_t = f(\textbf{s}_t)$$

The words in the decoder are always generated one at a time. We always use `<sos>` for the first input to the decoder, $y_1$, but for subsequent inputs, $\textbf{y}_{t>1}$, in training we will sometimes use the actual, ground truth next word $y_t$ (called *teacher forcing*, see a bit more info about it [here](https://machinelearningmastery.com/teacher-forcing-for-recurrent-neural-networks/)), and sometimes the previous word predicted by our decoder, $\hat{\textbf{y}}_{t-1}$.

When training our model, we always know the output sentence, so we stop generating when we reach the end of the output sequence. During inference it is common to keep generating words until the model outputs an `<eos>` token or until a given number of words have been generated.

In training, once we have our predicted output sentence, $\hat{\textbf{Y}} = \{ \hat{\textbf{y}}_1, \hat{\textbf{y}}_2, ..., \hat{\textbf{y}}_t \}$, we compare it against our target sentence, $\textbf{Y} = \{ \textbf{y}_1, \textbf{y}_2, ..., \textbf{y}_t \}$, to calculate our loss. We then use this loss to update all of the parameters in our model.

In our Seq2Seq encoder-decoder implementation in this lab we are using RNN stacks in encoder and decoder. Recall the general architecture of stacked RNNs from the lecture:

![](background_2.png)

In the lab, we are stacking two LSTM layers in both the encoder and the decoder, giving us the following equations defining each stack of two LSTMs:

$$\begin{align*}
(\textbf{h}_t^1, \textbf{c}_t^1) &= \text{LSTM}^1(e(x_t), (h_{t-1}^1, c_{t-1}^1))\\
(\textbf{h}_t^2, \textbf{c}_t^2) &= \text{LSTM}^2(\textbf{h}_t^1, (\textbf{h}_{t-1}^2, \textbf{c}_{t-1}^2))
\end{align*}$$

Note how only our hidden state from the first layer is passed as input to the second layer, and not the cell state.

## 3 Implementation

### 3.1 Preliminaries

To facilitate data preparation, install the following packages. **Please note** that refreshing the runtime is necessary for these packages to take effect.

In [1]:
# The portalocker package provides cross-platform file locking support for Python.
# File locking is essential for scenarios where multiple processes or threads need to access or modify the same file concurrently.
# Without file locking, race conditions can occur, leading to data corruption or inconsistencies.
# Portalocker helps prevent these issues by providing a mechanism for processes or threads to coordinate access to the file.
!pip install portalocker

zsh:1: command not found: pip


`uv pip install -U pip setuptools wheel`

`uv run python -m pip -V`

`uv run python -m spacy download en_core_web_sm`

`uv add for all other packages`

In [ ]:
# Import the clear_output function from IPython.display module for clearing Jupyter Notebook cell output
from IPython.display import clear_output

# Update Numpy
!pip install "numpy<2"
# This command installs or upgrades the TorchData library, which provides utilities and abstractions for handling datasets in PyTorch, including data loading and preprocessing.
!pip install -U torchdata
# This command installs or upgrades the spaCy library, which is a powerful natural language processing (NLP) library in Python. SpaCy provides various functionalities such as tokenization, POS tagging, dependency parsing, and named entity recognition (NER).
!pip install -U spacy
# This command downloads the English language model 'en_core_web_sm' provided by spaCy. This model includes pre-trained word vectors and various linguistic annotations, making it suitable for tasks such as tokenization, part-of-speech tagging, and syntactic parsing.
!python -m spacy download en_core_web_sm
# uv run python -m spacy download en_core_web_sm``

# This command downloads the German language model 'de_core_news_sm' provided by spaCy. Similar to the English model, it includes pre-trained word vectors and linguistic annotations tailored for German text processing tasks.
!python -m spacy download de_core_news_sm
#uv run python -m spacy download de_core_news_sm 

!pip install torchtext==0.16.2
# This command clears the output of the current cell in a Jupyter notebook environment. It can be useful for removing clutter and focusing on the most recent output or results.
clear_output()

In [1]:
# Import the main PyTorch library
import torch

# Import the torch.nn module, which contains neural network-related classes and functions
import torch.nn as nn

# Import the torch.optim module, which provides optimization algorithms for updating neural network parameters
import torch.optim as optim

# Import the DataLoader class from torch.utils.data module, used for loading datasets in mini-batches during training
from torch.utils.data import Dataset, DataLoader

# Import the pad_sequence function from torch.nn.utils.rnn module, used for padding sequences within a batch
from torch.nn.utils.rnn import pad_sequence

# Import the get_tokenizer function from torchtext.data.utils module, used for tokenizing text data
from torchtext.data.utils import get_tokenizer

# Import the build_vocab_from_iterator function from torchtext.vocab module, used for building vocabulary from tokenized text data
from torchtext.vocab import build_vocab_from_iterator

# Import the multi30k dataset and Multi30k class from torchtext.datasets module
from torchtext.datasets import multi30k, Multi30k

# Import the Iterable and List classes/types from typing module for type hinting
from typing import Iterable, List

# Import the spaCy library for natural language processing tasks
import spacy

# Import the NumPy library with the alias np for numerical operations
import numpy as np

# Import the random module for generating random numbers
import random

# Import the math module for mathematical functions and constants
import math

# Import the time module for working with time
import time

We'll set the random seeds for deterministic results.

In [2]:
SEED = 1234
# Setting a seed ensures reproducibility of results in random processes.
# By setting the seed to a fixed value, random number generation becomes deterministic,
# meaning that the same sequence of random numbers will be generated each time the code is run.

# Set the seed for Python's built-in random module
random.seed(SEED)

# Set the seed for NumPy's random number generator
np.random.seed(SEED)

# Set the seed for PyTorch's random number generator on CPU
torch.manual_seed(SEED)

if torch.cuda.is_available():
  # Set the seed for PyTorch's random number generator on GPU (if available)
  torch.cuda.manual_seed(SEED)
  # Ensure deterministic behavior for cuDNN (CUDA Deep Neural Network library) operations in PyTorch
  # This helps in ensuring reproducibility when using CUDA (GPU acceleration) for deep learning operations
  torch.backends.cudnn.deterministic = True


It's time to use some GPUs, to do that, we need to define a `torch.device`. This is used to tell pytorch to put all the tensors in our code on the GPU or not. We use the `torch.cuda.is_available()` function, which will return `True` if a GPU is detected on our computer. We pass this `device` to the iterator.

In [5]:
# Check if CUDA (GPU acceleration) is available on the system
device = torch.device('cuda' if torch.cuda.is_available() else 'mps')


To display GPU device details you are using, run the following command line.

**RUN ONLY WHEN CUDA IS AVAILABLE**

In [4]:
# Execute the shell command to display GPU information using nvidia-smi
!nvidia-smi

zsh:1: command not found: nvidia-smi


### 3.2 LSTM Implementation

**Note**: There is already an implementation of LSTMs in PyTorch (`nn.LSTM`), and also `nn.LSTMCell`. If we have a sequence length of 1, we could use nn.LSTMCell instead of `nn.LSTM`, as it is designed to handle a batch of inputs that aren't necessarily in a sequence. `nn.LSTMCell` is just a single cell, whereas `nn.LSTM` is a wrapper around potentially multiple cells.

##### LSTM Forward pass:

1. **Forget Gate**:
   
   The forget gate controls what information should be discarded or kept from the cell state.
   
   $\textbf{f}_t = \sigma(\textbf{U}_f \textbf{h}_{t-1} + \textbf{W}_f \textbf{x}_t)$
   
   $\textbf{k}_t = \textbf{c}_{t-1} \odot \textbf{f}_t$

2. **Input Gate (Add gate)**:
   
   The input gate controls what new information should be stored in the cell state.
   
   $\textbf{g}_t = \tanh(\textbf{U}_g \textbf{h}_{t-1} + \textbf{W}_g \textbf{x}_t)$
   
   $\textbf{i}_t = \sigma(\textbf{U}_i \textbf{h}_{t-1} + \textbf{W}_i \textbf{x}_t)$
   
   $\textbf{j}_t = \textbf{g}_t \odot \textbf{i}_t$

3. **Update Cell State**:
   
   The cell state is updated using the forget gate and the input gate.
   
   $\textbf{c}_t = \textbf{j}_t + \textbf{k}_t$

4. **Output Gate**:
   
   The output gate controls what information should be output from the cell state.
   
   $\textbf{o}_t = \sigma(\textbf{U}_o \textbf{h}_{t-1} + \textbf{W}_o \textbf{x}_t)$
   
   $\textbf{h}_t = \textbf{o}_t \odot \tanh(\textbf{c}_t)$

In these equations:
- $ \textbf{x}_t $ represents the input at time step $ t $.
- $\textbf{h}_{t-1}$ represents the hidden state at time step $t-1$.
- $\textbf{f}_t$, $\textbf{i}_t$, $\textbf{o}_t$ are the values of the forget gate, input gate, and output gate at time step $t$, respectively.
- $\textbf{c}_{t-1}$ represents the cell state at time step $t-1$.
- $\textbf{g}_t$ represents the candidate cell state at time step $t$.
- $\textbf{j}_t$ represents the input modulation (mask) gate at time step $t$.
- $\textbf{k}_t$ represents the forget gate modulation (mask) at time step $t$.
- $\textbf{c}_t$ represents the updated cell state at time step $t$.
- $\textbf{h}_t$ represents the hidden state at time step $t$.
- $\sigma$ represents the sigmoid activation function, $\odot$ represents element-wise multiplication, and $\tanh$ represents the hyperbolic tangent activation function.


In [ ]:
# Since nn.Module has the backward function built-in, we can implicitly utilise it by inheriting nn.Module.
# eliminating the need to implement anything other than the forward pass.
class LSTMLayer(nn.Module):
    def __init__(self, input_size, hidden_size):
        '''
          - Args:
                - input_size: The number of expected features in the input.
                - hidden_size: The number of features in the hidden state.
          - Functionality:
                - Initialises the LSTM layer with the specified input size, and hidden size.
                - Initialises weight and bias parameters.
        '''
        super(LSTMLayer, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        # Define parameters for a single LSTM layer
        # We mark the weight matrices as a parameter of the model using nn.Parameter
        # This is done to indicate that this tensor should be considered a model parameter.
        # So that backward function in pytorch consider these matrices during optimisation,
        # and to calculate the gradients with respect to them.
        self.W_f = nn.Parameter(torch.Tensor(input_size, hidden_size))
        self.U_f = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_f = nn.Parameter(torch.Tensor(hidden_size))

        self.W_i = nn.Parameter(torch.Tensor(input_size, hidden_size))
        self.U_i = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_i = nn.Parameter(torch.Tensor(hidden_size))

        self.W_g = nn.Parameter(torch.Tensor(input_size, hidden_size))
        self.U_g = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_g = nn.Parameter(torch.Tensor(hidden_size))

        self.W_o = nn.Parameter(torch.Tensor(input_size, hidden_size))
        self.U_o = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_o = nn.Parameter(torch.Tensor(hidden_size))

    def forward(self, input, h_prev, c_prev):
        # Concatenate input and previous hidden state
        ## INSERT YOUR CODE HERE ##
        # Forget gate
        # Now that you're becoming comfortable with matrix multiplication,
        # we have commented on what's happening with the sizes for only one equation.
        # We want you to comment on the other equations as well.
        # input: (batch_size, input_size) @ W_f: (input_size, hidden_size) -> (batch_size, hidden_size)
        # h_prev: (batch_size, hidden_size) @ U_f: (hidden_size, hidden_size) -> (batch_size, hidden_size)
        
        # input: (batch_size, input_size) @ W_f: (input_size, hidden_size) +
        # h_prev: (batch_size, hidden_size) @ U_f (hidden_size, hidden_size) +
        # self.b_f(hidden_size, ) -> (batch_size, hidden_size)
        f = torch.sigmoid(input @ self.W_f + h_prev @ self.U_f + self.b_f)

        # Element-wise multiplication requires same shape
        # f: (batch_size, hidden_size) * c_prev: (batch_size, hidden_size) -> (batch_size, hidden_size)
        k = f * c_prev #mask


        # Input gate (Add gate)
        # input: (batch_size, input_size) @ W_i: (input_size, hidden_size) +
        # h_prev: (batch_size, hidden_size) @ U_i: (hidden_size, hidden_size) +
        # b_i: (hidden_size, ) -> (batch_size, hidden_size)
        i = torch.sigmoid(input @ self.W_i + h_prev @ self.U_i + self.b_i)

        # input: (batch_size, input_size) @ W_g:(input_size, hidden_size) +
        # h_prev: (batch_size, hidden_size) @ U_g: (hidden_size, hidden_size) +
        # b_g: (hidden_size, ) -> (batch_size, hidden_size)
        g = torch.tanh(input @ self.W_g + h_prev @ self.U_g + self.b_g) # Candidate cell state

        # g: (batch_size, hidden_size) * i: (batch_size, hidden_size) -> (batch_size, hidden_size)
        j =  g * i #mask


        # input: (batch_size, input_size) @ W_o: (input_size, hidden_size) +
        # h_prev: (batch_size, hidden_size) @ U_o: (hidden_size, hidden_size) +
        # b_o: (hidden_size) -> (batch_size, hidden_size)
        # Output gate
        o = torch.sigmoid(input @ self.W_o + h_prev @ self.U_o + self.b_o)


        # Update cell state
        # j: (batch_size, hidden_size) + k: (batch_size, hidden_size) -> (batch_size, hidden_size)
        c_next = j + k

        # Update hidden state
        # o: (batch_size, hidden_size) * tanh(c_next): (batch_size, hidden_size) -> (batch_size, hidden_size)
        h_next = o * torch.tanh(c_next)
        ## END OF YOUR CODE ##

        return h_next, c_next

In [10]:
class StackLSTMLayers(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1):
        '''
          - Args:
                - input_size: The number of expected features in the input.
                - hidden_size: The number of features in the hidden state.
                - num_layers: Number of Stacked recurrent layers.
          - Functionality:
                - Initialises the LSTM layer with the specified input size, hidden size, and number of layers.
                - Initialises weight and bias parameters for each layer.
        '''
        super(StackLSTMLayers, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Create stack of LSTM layers if num_layers > 1
        self.layers = nn.ModuleList([LSTMLayer(input_size if i == 0 else hidden_size, hidden_size) for i in range(num_layers)])

    def forward(self, input, hidden=None):
        if hidden is None:
            # initialise hidden and cell states if not provided
            hidden = self.init_hidden(input.size(1))

        # Unpack hidden states
        hiddens, cells = hidden

        outputs = []

        # Iterate through each time step
        for input_t in input:
            # Iterate through each layer
            for layer_idx, layer in enumerate(self.layers):
                # Pass input through the current layer
                hiddens[layer_idx], cells[layer_idx] = layer(input_t, hiddens[layer_idx], cells[layer_idx])

                # Update input for the next layer (if any)
                if layer_idx < self.num_layers - 1:
                    input_t = hiddens[layer_idx]

            # Append output of current time step
            outputs.append(hiddens[-1])

        # Stack outputs along the sequence dimension
        outputs = torch.stack(outputs, dim=0)


        # Return outputs, hidden states, and cell states
        return outputs, (hiddens, cells)

    def init_hidden(self, batch_size):
        # initialise hidden and cell states for each layer
        hiddens = [torch.zeros(batch_size, self.hidden_size, device=device) for _ in range(self.num_layers)]
        cells = [torch.zeros(batch_size, self.hidden_size, device=device) for _ in range(self.num_layers)]
        return hiddens, cells

As always, let's test our neural network using a dummy dataset to ensure that everything is working as intended before committing to training on a real dataset.

In [11]:
# Example usage
input_size = 10
hidden_size = 5
sequence_length = 3
batch_size = 2
stack_lstm_layers = 3

# initialise the stack of LSTM layers model
lstm = StackLSTMLayers(input_size, hidden_size, num_layers = stack_lstm_layers).to(device)

# Generate sample input data
input_data = torch.randn(sequence_length, batch_size, input_size)


# Get the output and updated hidden and cell states for the current time step
output, (hidden, cell) = lstm(input_data.to(device))

print(f"Hidden states number: {len(hidden)}. And each hidden state has shape: {hidden[0].shape}")
print(f"Cell states number: {len(cell)}. And each cell state has shape {cell[0].shape}")
print(f"Output shape: {output.shape}")

Hidden states number: 3. And each hidden state has shape: torch.Size([2, 5])
Cell states number: 3. And each cell state has shape torch.Size([2, 5])
Output shape: torch.Size([3, 2, 5])


In [12]:
#print the LSTM model's architecture (skeleton)
lstm

StackLSTMLayers(
  (layers): ModuleList(
    (0-2): 3 x LSTMLayer()
  )
)

## 4 Building the Encoder-Decoder Model

We'll be building our model in three parts. The encoder, the decoder and a Seq2Seq Encoder-Decoder model that encapsulates the encoder and decoder and will provide a way to interface with each.

### 4.1 Encoder

First, the encoder which is (recall from above) a two-layer LSTM.

For two-layer LSTMs in the encoder, we get:

$$\begin{align*}
(\textbf{h}_t^1, \textbf{c}_t^1) &= \text{EncoderLSTM}^1(e(\textbf{x}_t), (\textbf{h}_{t-1}^1, \textbf{c}_{t-1}^1))\\
(\textbf{h}_t^2, \textbf{c}_t^2) &= \text{EncoderLSTM}^2(\textbf{h}_t^1, (\textbf{h}_{t-1}^2, \textbf{c}_{t-1}^2))
\end{align*}$$

We create this in code by making an `Encoder` module, which requires we inherit from `torch.nn.Module` and use the `super().__init__()` as some boilerplate code. The encoder takes the following arguments:
- `input_dim` is the size/dimensionality of the one-hot vectors that will be input to the encoder. This is equal to the input (source) vocabulary size.
- `emb_dim` is the dimensionality of the embedding layer. This layer converts the one-hot vectors into dense vectors with `emb_dim` dimensions.
- `hid_dim` is the dimensionality of the hidden and cell states.
- `n_layers` is the number of layers in the RNN.
- `dropout` is the amount of dropout to use. This is a regularization parameter to prevent overfitting. Check out [this](https://www.coursera.org/lecture/deep-neural-network/understanding-dropout-YaGbR) for more details about dropout.

We are not going into details about the embedding layer in this lab. All we need to know is that there is a step before the words - technically, the indexes of the words - are passed into the RNN, where the words are transformed into vectors. Refer to the lecture slides and the Jurafsky and Martin textbook for details.

The embedding layer is created using `nn.Embedding`, and for the LSTM we use our custom implementation  `LSTM` and a dropout layer with `nn.Dropout`. Check the PyTorch [documentation](https://pytorch.org/docs/stable/nn.html) for more about these.

One thing to note is that the `dropout` argument to the LSTM is how much dropout to apply between the layers of a multi-layer RNN, i.e. between the hidden states output from layer $l$ and those same hidden states being used for the input of layer $l+1$.

In the `forward` method, we pass in the source sentence, $\textbf{X}$, which is converted into dense vectors using the `embedding` layer, and then dropout is applied. These embeddings are then passed into the RNN. As we pass a whole sequence to the RNN, it will automatically do the recurrent calculation of the hidden states over the whole sequence for us. Notice that we do not pass an initial hidden or cell state to the RNN. This is because, as noted in the [documentation](https://pytorch.org/docs/stable/nn.html#torch.nn.LSTM), that if no hidden/cell state is passed to the RNN, it will automatically create an initial hidden/cell state as a tensor of all zeros.

The LSTM returns: `outputs` (the top-layer hidden state for each time-step), `hidden` (the final hidden state for each layer, $\textbf{h}_t$, stacked on top of each other) and `cell` (the final cell state for each layer, $\textbf{c}_t$, stacked on top of each other).

As we only need the final hidden and cell states (to make our context vector), `forward` only returns `hidden` and `cell`.

In [14]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()

        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_dim, emb_dim)

        self.rnn = StackLSTMLayers(emb_dim, hid_dim, n_layers)

        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        ## INSERT YOUR CODE HERE ##
        #src = [src len, batch size]

        embedded = self.dropout(self.embedding(src)) ## UNDERSTAND THIS

        #embedded = [src len, batch size, emb dim]

        outputs, (hidden, cell) = self.rnn(embedded)

        #outputs = [src len, batch size, hid dim * n directions]
        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]

        #outputs are always from the top hidden layer
        ## END OF YOUR CODE ##
        return hidden, cell

### 4.2 Decoder

Next, we'll build our decoder, which will also be a 2-layer (4 in the paper) LSTM.

The `Decoder` class does a single step of decoding, i.e. it outputs single token per time-step. The first layer will receive a hidden and cell state from the previous time-step, $(\textbf{s}_{t-1}^1, \textbf{c}_{t-1}^1)$, and feeds it through the LSTM with the current embedded token, $y_t$, to produce a new hidden and cell state, $(\textbf{s}_t^1, \textbf{c}_t^1)$. The subsequent layers will use the hidden state from the layer below, $\textbf{s}_t^{l-1}$, and the previous hidden and cell states from their layer, $(\textbf{s}_{t-1}^l, \textbf{c}_{t-1}^l)$. This provides equations very similar to those in the encoder.

$$\begin{align*}
(\textbf{s}_t^1, \textbf{c}_t^1) = \text{DecoderLSTM}^1(d(\textbf{y}_t), (\textbf{s}_{t-1}^1, \textbf{c}_{t-1}^1))\\
(\textbf{s}_t^2, \textbf{c}_t^2) = \text{DecoderLSTM}^2(\textbf{s}_t^1, (\textbf{s}_{t-1}^2, \textbf{c}_{t-1}^2))
\end{align*}$$

Remember that the initial hidden and cell states to our decoder are our context vectors, which are the final hidden and cell states of our encoder from the same layer, i.e. $(\textbf{s}_0^l,\textbf{c}_0^l)=z^l=(\textbf{h}_t^l, \textbf{c}_t^l)$.

We then pass the hidden state from the top layer of the RNN, $s_t^L$, through a linear layer, $f$, to make a prediction of what the next token in the target (output) sequence should be, $\hat{\textbf{y}}_{t+1}$.

$$\hat{\textbf{y}}_{t+1} = f(\textbf{s}_t^L)$$

The arguments and initialisation are similar to the `Encoder` class, except we now have an `output_dim` which is the size of the vocabulary for the output/target. There is also the addition of the `Linear` layer, used to make the predictions from the top layer hidden state.

Within the `forward` method, we accept a batch of input tokens, previous hidden states and previous cell states. As we are only decoding one token at a time, the input tokens will always have a sequence length of 1. We `unsqueeze` the input tokens to add a sentence length dimension of 1. Then, similar to the encoder, we pass through an embedding layer and apply dropout. This batch of embedded tokens is then passed into the RNN with the previous hidden and cell states. This produces an `output` (hidden state from the top layer of the RNN), a new `hidden` state (one for each layer, stacked on top of each other) and a new `cell` state (also one per layer, stacked on top of each other). We then pass the `output` (after getting rid of the sentence length dimension) through the linear layer to receive our `prediction`. We then return the `prediction`, the new `hidden` state and the new `cell` state.

In [15]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):

        super().__init__()

        self.output_dim = output_dim
        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(output_dim, emb_dim)

        self.rnn = StackLSTMLayers(emb_dim, hid_dim, n_layers)

        self.fc_out = nn.Linear(hid_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        ## INSERT YOUR CODE HERE ##
        #input = [batch size]
        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]

        #n directions in the decoder will both always be 1, therefore:
        #hidden = [n layers, batch size, hid dim]
        #context = [n layers, batch size, hid dim]

        input = input.unsqueeze(0)

        #input = [1, batch size]

        embedded = self.dropout(self.embedding(input))

        #embedded = [1, batch size, emb dim]

        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))

        #output = [seq len, batch size, hid dim * n directions]
        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]

        #seq len and n directions will always be 1 in the decoder, therefore:
        #output = [1, batch size, hid dim]
        #hidden = [n layers, batch size, hid dim]
        #cell = [n layers, batch size, hid dim]

        prediction = self.fc_out(output.squeeze(0))

        #prediction = [batch size, output dim]
        ## END OF YOUR CODE ##
        return prediction, hidden, cell

### 4.3 Seq2Seq Encoder-Decoder

For the final part of the implementation, we'll implement the Seq2Seq Encoder-Decoder model. This will handle:
- receiving the input/source sentence
- using the encoder to produce the context vectors
- using the decoder to produce the predicted output/target sentence

The `Seq2Seq Encoder-Decoder` model takes in an `Encoder`, `Decoder`, and a `device` (used to place tensors on the GPU, if it exists).

For this implementation, we have to ensure that the number of layers and the hidden (and cell) dimensions are equal in the `Encoder` and `Decoder`. This is not always the case, we do not necessarily need the same number of layers or the same hidden dimension sizes in a sequence-to-sequence model. However, if we did something like having a different number of layers then we would need to make decisions about how this is handled. For example, if our encoder has 2 layers and our decoder only has 1, how is this handled? Do we average the two context vectors output by the decoder? Do we pass both through a linear layer? Do we only use the context vector from the highest layer? Etc.

Our `forward` method takes the source sentence, target sentence and a teacher-forcing ratio. The teacher forcing ratio is used when training our model. When decoding, at each time-step we will predict what the next token in the target sequence will be from the previous tokens decoded, $\hat{\textbf{y}}_{t+1}=f(\textbf{s}_t^L)$. With probability equal to the teacher forcing ratio (`teacher_forcing_ratio`) we will use the actual ground-truth next token in the sequence as the input to the decoder during the next time-step. However, with probability `1 - teacher_forcing_ratio`, we will use the token that the model predicted as the next input to the model, even if it doesn't match the actual next token in the sequence.  

The first thing we do in the `forward` method is to create an `outputs` tensor that will store all of our predictions, $\hat{\textbf{Y}}$.

We then feed the input/source sentence, `src`, into the encoder and receive out final hidden and cell states.

The first input to the decoder is the start of sequence (`<sos>`) token. As our `trg` tensor already has the `<sos>` token appended (all the way back when we defined the `init_token` in our `TRG` field) we get our $y_1$ by slicing into it. We know how long our target sentences should be (`max_len`), so we loop that many times. The last token input into the decoder is the one **before** the `<eos>` token - the `<eos>` token is never input into the decoder.

During each iteration of the loop, we:
- pass the input, previous hidden and previous cell states ($\textbf{y}_t, \textbf{s}_{t-1}, \textbf{c}_{t-1}$) into the decoder
- receive a prediction, next hidden state and next cell state ($\hat{\textbf{y}}_{t+1}, \textbf{s}_{t}, \textbf{c}_{t}$) from the decoder
- place our prediction, $\hat{\textbf{y}}_{t+1}$ `output` in our tensor of predictions, $\hat{\textbf{Y}}$`outputs`
- decide if we are going to "teacher force" or not
    - if we do, the next `input` is the ground-truth next token in the sequence, $\textbf{y}_{t+1}$`trg[t]`
    - if we don't, the next `input` is the predicted next token in the sequence, $\hat{\textbf{y}}_{t+1}$`top1`, which we get by doing an `argmax` over the output tensor
    
Once we've made all of our predictions, we return our tensor full of predictions, $\hat{Y}$`outputs`.

**Note**: our decoder loop starts at 1, not 0. This means the 0th element of our `outputs` tensor remains all zeros. So our `trg` and `outputs` look something like:

$$\begin{align*}
\text{trg} = [<sos>, &\textbf{y}_1, \textbf{y}_2, \textbf{y}_3, <eos>]\\
\text{outputs} = [0, &\hat{\textbf{y}}_1, \hat{\textbf{y}}_2, \hat{\textbf{y}}_3, <eos>]
\end{align*}$$

Later on when we calculate the loss, we cut off the first element of each tensor to get:

$$\begin{align*}
\text{trg} = [&\textbf{y}_1, \textbf{y}_2, \textbf{y}_3, <eos>]\\
\text{outputs} = [&\hat{\textbf{y}}_1, \hat{\textbf{y}}_2, \hat{\textbf{y}}_3, <eos>]
\end{align*}$$

In [16]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

        assert encoder.hid_dim == decoder.hid_dim, \
            "Hidden dimensions of encoder and decoder must be equal!"
        assert encoder.n_layers == decoder.n_layers, \
            "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio = 0.5):

        #src = [src len, batch size]
        #trg = [trg len, batch size]
        #teacher_forcing_ratio is probability to use teacher forcing
        #e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time

        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        #tensor to store decoder outputs
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        #last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)

        #first input to the decoder is the <sos> tokens
        input = trg[0,:]

        for t in range(1, trg_len):

            #insert input token embedding, previous hidden and previous cell states
            #receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)

            #place predictions in a tensor holding predictions for each token
            outputs[t] = output

            #decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio

            #get the highest predicted token from our predictions
            top1 = output.argmax(1)

            #if teacher forcing, use actual next token as next input
            #if not, use predicted token
            input = trg[t] if teacher_force else top1

        return outputs

## 5 Preparing Data

We'll be coding up the models in PyTorch and using torchtext to help us do all of the pre-processing required. We'll also be using spaCy to assist in the tokenization of the data.

Next, we'll create the tokenizers. A tokenizer is used to turn a string containing a sentence into a list of individual tokens that make up that string, e.g. "good morning!" becomes ["good", "morning", "!"]. We'll start talking about the sentences being a sequence of tokens from now, instead of saying they're a sequence of words. What's the difference? Well, "good" and "morning" are both words and tokens, but "!" is a token, not a word.

spaCy has model for each language ("de_core_news_sm" for German and "en_core_web_sm" for English) which need to be loaded so we can access the tokenizer of each model.

**Note**: the models must first be downloaded using the following commands:

Next, we download and load the train, validation and test data.

The dataset we'll be using is the [Multi30k dataset](https://github.com/multi30k/dataset). This is a dataset with ~30,000 parallel English, German and French sentences, each with ~12 words per sentence.

`split` specifies which splits we want to download and load and `language_pair` specifies which languages to use as the source and target (source goes first).

In [17]:
# We need to modify the URLs for the dataset since the links to the original dataset are broken
# Refer to https://github.com/pytorch/text/issues/1756#issuecomment-1163664163 for more info
multi30k.URL["train"] = "https://raw.githubusercontent.com/neychev/small_DL_repo/master/datasets/Multi30k/training.tar.gz"
multi30k.URL["valid"] = "https://raw.githubusercontent.com/neychev/small_DL_repo/master/datasets/Multi30k/validation.tar.gz"

# NOTE: There is a known issue with the test set path for the Multi30k dataset where the original path may not work as expected.
#multi30k.URL["test"] = r"https://raw.githubusercontent.com/neychev/small_DL_repo/master/datasets/Multi30k/mmt16_task1_test.tar.gz"
#Update hash since there is a discrepancy between user hosted test split and that of the test split in the original dataset
#multi30k.MD5["test"] = "6d1ca1dba99e2c5dd54cae1226ff11c2551e6ce63527ebb072a1f70f72a5cd36"

SRC_LANGUAGE = 'de'
TGT_LANGUAGE = 'en'

# Place-holders
token_transform = {}
vocab_transform = {}

In [18]:
token_transform[SRC_LANGUAGE] = get_tokenizer('spacy', language='de_core_news_sm')
token_transform[TGT_LANGUAGE] = get_tokenizer('spacy', language='en_core_web_sm')

In [19]:
#There is a problem with test iter, so
train_iter, valid_iter = Multi30k(split = ('train', 'valid'),
                                             language_pair = (SRC_LANGUAGE, TGT_LANGUAGE))


Next, we'll build the *vocabulary* for the source and target languages. The vocabulary is used to associate each unique token with an index (an integer). The vocabularies of the source and target languages are distinct.

Using the `min_freq` argument, we only allow tokens that appear at least two times to appear in our vocabulary. Tokens that appear only once are converted into an `<unk>` (unknown) token.

It is important to note that our vocabulary should only be built from the training set and not the validation/test set. This prevents "information leakage" into our model, giving us artifically inflated validation/test scores.

In [20]:
# helper function to yield list of tokens
def yield_tokens(data_iter: Iterable, language: str) -> List[str]:
    language_index = {SRC_LANGUAGE: 0, TGT_LANGUAGE: 1}

    for data_sample in data_iter:
          yield token_transform[language](data_sample[language_index[language]])

# Define special symbols and indices
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
# Make sure the tokens are in order of their indices to properly insert them in vocab
special_symbols = ['<unk>', '<pad>', '<bos>', '<eos>']

for ln in [SRC_LANGUAGE, TGT_LANGUAGE]:
    # Create torchtext's Vocab object
    vocab_transform[ln] = build_vocab_from_iterator(yield_tokens(train_iter, ln),
                                                    min_freq=2,
                                                    specials=special_symbols,
                                                    special_first=True)

# Set ``UNK_IDX`` as the default index. This index is returned when the token is not found.
# If not set, it throws ``RuntimeError`` when the queried token is not found in the Vocabulary.
for ln in [SRC_LANGUAGE, TGT_LANGUAGE]:
  vocab_transform[ln].set_default_index(UNK_IDX)

/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/torch/utils/data/datapipes/iter/combining.py:337: UserWarning: Some child DataPipes are not exhausted when __iter__ is called. We are resetting the buffer and each child DataPipe will read from the start again.
  warnings.warn("Some child DataPipes are not exhausted when __iter__ is called. We are resetting "


In [21]:
print(f"Unique tokens in source (de) vocabulary: {len(vocab_transform[SRC_LANGUAGE].vocab)}")
print(f"Unique tokens in target (en) vocabulary: {len(vocab_transform[TGT_LANGUAGE].vocab)}")

Unique tokens in source (de) vocabulary: 8014
Unique tokens in target (en) vocabulary: 6191


The data iterator yields a pair of raw strings. We need to convert these string pairs into the batched tensors that can be processed by our Seq2Seq (Encoder-Decoder) network. Below we define our collate function that converts a batch of raw strings into batch tensors that can be fed directly into our model.

In [22]:
# helper function to club together sequential operations
def sequential_transforms(*transforms):
    def func(txt_input):
        for transform in transforms:
            txt_input = transform(txt_input)
        return txt_input
    return func

# function to add BOS/EOS and create tensor for input sequence indices
def tensor_transform(token_ids: List[int]):
    return torch.cat((torch.tensor([BOS_IDX]),
                      torch.tensor(token_ids),
                      torch.tensor([EOS_IDX])))

# ``src`` and ``tgt`` language text transforms to convert raw strings into tensors indices
text_transform = {}
for ln in [SRC_LANGUAGE, TGT_LANGUAGE]:
    text_transform[ln] = sequential_transforms(token_transform[ln], #Tokenization
                                               vocab_transform[ln], #Numericalization
                                               tensor_transform) # Add BOS/EOS and create tensor


# function to collate data samples into batch tensors
def collate_fn(batch):
    src_batch, tgt_batch = [], []
    for src_sample, tgt_sample in batch:
        src_batch.append(text_transform[SRC_LANGUAGE](src_sample.rstrip("\n")))
        tgt_batch.append(text_transform[TGT_LANGUAGE](tgt_sample.rstrip("\n")))

    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX)
    return src_batch, tgt_batch

## 6 Training the Seq2Seq Encoder-Decoder Model

Now we have our model implemented, we can begin training it.

First, we'll initialise our model. As mentioned before, the input and output dimensions are defined by the size of the vocabulary. The embedding dimensions and dropout for the encoder and decoder can be different, but the number of layers and the size of the hidden/cell states must be the same.

We then define the encoder, decoder and then our Seq2Seq Encoder-Decoder model, which we place on the `device`.

### 6.1 Initialise Seq2Seq Model

In [23]:
INPUT_DIM = len(vocab_transform[SRC_LANGUAGE])
OUTPUT_DIM = len(vocab_transform[TGT_LANGUAGE])
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.2
DEC_DROPOUT = 0.2
BATCH_SIZE = 128

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)

model = Seq2Seq(enc, dec, device).to(device)

### 6.2 Weights Initialisation

Next up is initialising the weights of our model. In the paper they state they initialise all weights from a uniform distribution between -0.08 and +0.08, i.e. $\mathcal{U}(-0.08, 0.08)$.

We initialise weights in PyTorch by creating a function which we `apply` to our model. When using `apply`, the `init_weights` function will be called on every module and sub-module within our model. For each module we loop through all of the parameters and sample them from a uniform distribution with `nn.init.uniform_`.

In [24]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(8014, 256)
    (rnn): StackLSTMLayers(
      (layers): ModuleList(
        (0-1): 2 x LSTMLayer()
      )
    )
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(6191, 256)
    (rnn): StackLSTMLayers(
      (layers): ModuleList(
        (0-1): 2 x LSTMLayer()
      )
    )
    (fc_out): Linear(in_features=512, out_features=6191, bias=True)
    (dropout): Dropout(p=0.2, inplace=False)
  )
)

### 6.3 Count the Number of Weights in the model

We also define a function that will calculate the number of trainable parameters in the model.

In [25]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

The model has 14,160,687 trainable parameters


In [26]:
for name, _ in model.named_parameters():
  print(name)

encoder.embedding.weight
encoder.rnn.layers.0.W_f
encoder.rnn.layers.0.U_f
encoder.rnn.layers.0.b_f
encoder.rnn.layers.0.W_i
encoder.rnn.layers.0.U_i
encoder.rnn.layers.0.b_i
encoder.rnn.layers.0.W_g
encoder.rnn.layers.0.U_g
encoder.rnn.layers.0.b_g
encoder.rnn.layers.0.W_o
encoder.rnn.layers.0.U_o
encoder.rnn.layers.0.b_o
encoder.rnn.layers.1.W_f
encoder.rnn.layers.1.U_f
encoder.rnn.layers.1.b_f
encoder.rnn.layers.1.W_i
encoder.rnn.layers.1.U_i
encoder.rnn.layers.1.b_i
encoder.rnn.layers.1.W_g
encoder.rnn.layers.1.U_g
encoder.rnn.layers.1.b_g
encoder.rnn.layers.1.W_o
encoder.rnn.layers.1.U_o
encoder.rnn.layers.1.b_o
decoder.embedding.weight
decoder.rnn.layers.0.W_f
decoder.rnn.layers.0.U_f
decoder.rnn.layers.0.b_f
decoder.rnn.layers.0.W_i
decoder.rnn.layers.0.U_i
decoder.rnn.layers.0.b_i
decoder.rnn.layers.0.W_g
decoder.rnn.layers.0.U_g
decoder.rnn.layers.0.b_g
decoder.rnn.layers.0.W_o
decoder.rnn.layers.0.U_o
decoder.rnn.layers.0.b_o
decoder.rnn.layers.1.W_f
decoder.rnn.layers.1.U_f


### 6.4 Define the optimiser and pass the model weights to it

We define our optimiser, which we use to update our parameters in the training loop. Here, we'll use Adam.

Adam (Adaptive Moment Estimation) is an optimisation algorithm commonly used in training neural networks.  In simple terms, Adam adjusts the learning rate for each parameter individually, based on the gradient and the moving averages of past gradients and squared gradients. This adaptive learning rate helps Adam converge quickly and efficiently, making it the most used optimisation algorithm in deep learning.
Check out [this](http://ruder.io/optimizing-gradient-descent/) post for information about different optimizers.

In [27]:
optimizer = optim.Adam(model.parameters())

### 6.5 Define the loss function

Next, we define our loss function. The `CrossEntropyLoss` function calculates both the log softmax as well as the negative log-likelihood of our predictions.

Our loss function calculates the average loss per token, however by passing the index of the `<pad>` token as the `ignore_index` argument we ignore the loss whenever the target token is a padding token.

In [28]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

### 6.6 Define Training Loop

Next, we'll define our training loop.

First, we'll set the model into "training mode" with `model.train()`. This will turn on dropout (and batch normalization, which we aren't using) and then iterate through our data iterator.

As stated before, our decoder loop starts at 1, not 0. This means the 0th element of our `outputs` tensor remains all zeros. So our `trg` and `outputs` look something like:

$$\begin{align*}
\text{trg} = [<sos>, &y_1, y_2, y_3, <eos>]\\
\text{outputs} = [0, &\hat{y}_1, \hat{y}_2, \hat{y}_3, <eos>]
\end{align*}$$

Here, when we calculate the loss, we cut off the first element of each tensor to get:

$$\begin{align*}
\text{trg} = [&y_1, y_2, y_3, <eos>]\\
\text{outputs} = [&\hat{y}_1, \hat{y}_2, \hat{y}_3, <eos>]
\end{align*}$$

At each iteration:
- get the source and target sentences from the batch, $X$ and $Y$
- zero the gradients calculated from the last batch
- feed the source and target into the model to get the output, $\hat{Y}$
- as the loss function only works on 2d inputs with 1d targets we need to flatten each of them with `.view`
- we slice off the first column of the output and target tensors as mentioned above
- calculate the gradients with `loss.backward()`
- clip the gradients to prevent them from exploding (a common issue in RNNs)
- update the parameters of our model by doing an optimizer step
- sum the loss value to a running total

Finally, we return the loss that is averaged over all batches.

In [29]:
def train(model, iterator, optimizer, criterion, clip):

    model.train()

    epoch_loss = 0

    num_batches = 0
    dataloader = DataLoader(iterator, batch_size=BATCH_SIZE, collate_fn=collate_fn)

    for src, trg in dataloader:

        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()

        output = model(src, trg)

        #trg = [trg len, batch size]
        #output = [trg len, batch size, output dim]

        output_dim = output.shape[-1]

        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)

        #trg = [(trg len - 1) * batch size]
        #output = [(trg len - 1) * batch size, output dim]

        loss = criterion(output, trg)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    return epoch_loss / max(1, num_batches)

### 6.7 Define Evaluation Loop

Our evaluation loop is similar to our training loop, however as we aren't updating any parameters we don't need to pass an optimizer or a clip value.

We must remember to set the model to evaluation mode with `model.eval()`. This will turn off dropout (and batch normalization, if used).

We use the `with torch.no_grad()` block to ensure no gradients are calculated within the block. This reduces memory consumption and speeds things up.

The iteration loop is similar (without the parameter updates), however we must ensure we turn teacher forcing off for evaluation. This will cause the model to only use its own predictions to make further predictions within a sentence, which mirrors how it would be used in deployment.

In [30]:
def evaluate(model, iterator, criterion):

    model.eval()

    epoch_loss = 0

    num_batches = 0
    dataloader = DataLoader(iterator, batch_size=BATCH_SIZE, collate_fn=collate_fn)

    with torch.no_grad():

        for src, trg in dataloader:
            src = src.to(device)
            trg = trg.to(device)

            output = model(src, trg, 0) #turn off teacher forcing

            #trg = [trg len, batch size]
            #output = [trg len, batch size, output dim]

            output_dim = output.shape[-1]

            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)

            #trg = [(trg len - 1) * batch size]
            #output = [(trg len - 1) * batch size, output dim]

            loss = criterion(output, trg)

            epoch_loss += loss.item()
            num_batches += 1

    return epoch_loss / max(1, num_batches)

### 6.8 Start the Training Process

Next, we'll create a function that we'll use to tell us how long an epoch takes.

In [31]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

We can finally start training our model!

At each epoch, we'll be checking if our model has achieved the best validation loss so far. If it has, we'll update our best validation loss and save the parameters of our model (called `state_dict` in PyTorch). Then, when we come to test our model, we'll use the saved parameters used to achieve the best validation loss.

We'll be printing out both the loss and the perplexity at each epoch. It is easier to see a change in perplexity than a change in loss as the numbers are much bigger.

perplexity quantifies how surprised a language model is when predicting the next word in a sequence. A lower perplexity value indicates that the model is more confident and less surprised by the actual words in the sequence.

In [32]:
N_EPOCHS = 10
CLIP = 1

best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):

    start_time = time.time()

    train_loss = train(model, train_iter, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, valid_iter, criterion)

    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'lstm-seq2seq-model.pt')

    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')

/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/torch/utils/data/datapipes/iter/combining.py:337: UserWarning: Some child DataPipes are not exhausted when __iter__ is called. We are resetting the buffer and each child DataPipe will read from the start again.
  warnings.warn("Some child DataPipes are not exhausted when __iter__ is called. We are resetting "


Epoch: 01 | Time: 1m 2s
	Train Loss: 5.273 | Train PPL: 195.042
	 Val. Loss: 5.018 |  Val. PPL: 151.044
Epoch: 02 | Time: 0m 59s
	Train Loss: 4.634 | Train PPL: 102.959
	 Val. Loss: 4.997 |  Val. PPL: 147.983
Epoch: 03 | Time: 0m 58s
	Train Loss: 4.353 | Train PPL:  77.680
	 Val. Loss: 4.929 |  Val. PPL: 138.248
Epoch: 04 | Time: 0m 58s
	Train Loss: 4.148 | Train PPL:  63.277
	 Val. Loss: 4.689 |  Val. PPL: 108.746
Epoch: 05 | Time: 0m 59s
	Train Loss: 3.985 | Train PPL:  53.800
	 Val. Loss: 4.604 |  Val. PPL:  99.885
Epoch: 06 | Time: 0m 59s
	Train Loss: 3.843 | Train PPL:  46.653
	 Val. Loss: 4.573 |  Val. PPL:  96.788
Epoch: 07 | Time: 0m 59s
	Train Loss: 3.766 | Train PPL:  43.188
	 Val. Loss: 4.468 |  Val. PPL:  87.209
Epoch: 08 | Time: 0m 59s
	Train Loss: 3.636 | Train PPL:  37.924
	 Val. Loss: 4.436 |  Val. PPL:  84.411
Epoch: 09 | Time: 0m 59s
	Train Loss: 3.543 | Train PPL:  34.560
	 Val. Loss: 4.402 |  Val. PPL:  81.587
Epoch: 10 | Time: 0m 58s
	Train Loss: 3.436 | Train PPL:

After training our model and saved it at best validation loss, we can load the parameters (`state_dict`) and you can start using it for inference.

In [ ]:
model.load_state_dict(torch.load('lstm-seq2seq-model.pt'))

loss = evaluate(model, valid_iter, criterion)

print(f'| Loss: {loss:.3f} | PPL: {math.exp(loss):7.3f} |')


## 7 Week 5 Submission Task: Use the encoder-decoder framework for a language modeling task (predicting the next token).

This task requires you to prepare, preprocess, and tokenize your own data. Then, you'll use the same encoder-decoder implementation as above. For simplicity and to stay within the Colab compute budget, select any text passages you are interested in (hopefully long enough), make a split into training, validation, and test sets, and test how accurately your model completes sentences in your test set after training.

Next token prediction is essentially a shifted version of the input. For example, let's assume we have this sentence: "I love Neural Networks." Let's also assume that each word represents a token. Then, preparing input-output pairs for next token prediction (language modeling) will look like this:
- Input: "I love Neural"
- Output: "love Neural Networks"

It's similar to translation tasks in the sense that we transform one sequence into another.

Aim: create the code as above and in the last cell, include 10 test inputs, run them through the network and print out the outputs.

What to submit:
- your notebook and any datafiles etc. that your code depends on
- NB: include (in the notebook) any pip install statements that your code relies on
- ensure that before submitting you test the code by restarting the runtime and then hitting run all

**Hints**:
1. Unlike translation which needs source and target languages, language modeling predicts the next token in the same language. Look at how the data loading and preprocessing needs to change.
2. The sequence pairs in language modeling are related: if your input is [w1, w2, w3], your target should be [w2, w3, w4]. How would you modify the dataset class to create these pairs?
3. Notice that we don't need separate vocabularies for source and target languages anymore - how does this simplify the vocabulary creation?


## Resources

- Speech and Language Processing (3rd ed. draft) Dan Jurafsky and James H. Martin, [Chapter 13](https://web.stanford.edu/~jurafsky/slp3/13.pdf).

- Seq2Seq paper: [Sutskever et al., 2014](https://arxiv.org/abs/1409.3215).